# Baseline 1 — Markov Baseline

Sanity floor. No learned parameters, no GPU, no training loop.

**Idea:** For each training example, record `(last_cell_of_prefix → dest_region)`. At test time, take the last cell of the prefix, look up the most frequent destinations from training, and return the top-K.

**Fallback:** If a last cell was never seen in training, use the global destination frequency distribution.

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
from math import radians, cos, sin, asin, sqrt
import numpy as np
import torch

DATA_DIR = Path('porto_data_bundle')

# Verify centroid file exists — run build_centroids.py first if not
assert (DATA_DIR / 'cell_centroids.pt').exists(), \
    'cell_centroids.pt not found. Run build_centroids.py first.'

print('Setup OK.')

Setup OK.


## Step 1 — Build transition counts from training data

In [2]:
# Scan all 50 train shards.
# For each example we record:
#   transition_counts[last_cell][dest] += 1   → for per-cell top-K prediction
#   global_dest_freq[dest]            += 1   → for fallback when last_cell unseen

shard_paths = sorted((DATA_DIR / 'supervised_shards' / 'train').glob('train_*.pt'))
print(f'Scanning {len(shard_paths)} training shards...')

transition_counts = defaultdict(Counter)  # {last_cell: Counter({dest: count})}
global_dest_freq  = Counter()             # {dest: count} across all training examples

for i, path in enumerate(shard_paths):
    shard = torch.load(path, map_location='cpu', weights_only=False)
    for ex in shard:
        last_cell = ex['prefix_region_seq'][-1]   # last visited region in prefix
        dest      = ex['dest_region']
        transition_counts[last_cell][dest] += 1
        global_dest_freq[dest] += 1
    if (i + 1) % 10 == 0:
        print(f'  {i + 1}/{len(shard_paths)} shards done')

print(f'\nUnique last cells seen in training : {len(transition_counts)}')
print(f'Unique destination cells seen      : {len(global_dest_freq)}')

Scanning 50 training shards...
  10/50 shards done
  20/50 shards done
  30/50 shards done
  40/50 shards done
  50/50 shards done

Unique last cells seen in training : 5389
Unique destination cells seen      : 4978


## Step 2 — Build predictor (precompute sorted top-K lists)

In [3]:
# Keep top-10 to support Recall@10
TOP_K = 10

# For each last_cell, sort destinations by count descending and keep top-K.
# This avoids sorting at eval time, which matters since we evaluate ~1M examples.
cell_topk = {
    last_cell: [dest for dest, _ in counts.most_common(TOP_K)]
    for last_cell, counts in transition_counts.items()
}

# Global fallback list for cells not seen in training
global_topk = [dest for dest, _ in global_dest_freq.most_common(TOP_K)]

print(f'Predictor ready.')
print(f'Cells with transition data : {len(cell_topk)}')
print(f'Global fallback top-5      : {global_topk[:5]}')

Predictor ready.
Cells with transition data : 5389
Global fallback top-5      : [5171, 2429, 2300, 2360, 2362]


## Step 3 — Evaluation helpers

In [4]:
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in km between two (lat, lon) points."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * R * asin(sqrt(a))


def evaluate(shard_paths, cell_topk, global_topk, centroids, k_values=(1, 5, 10)):
    """
    Evaluate the Markov predictor on a set of shards.

    Haversine is computed using the top-1 predicted cell mapped to its
    centroid lat/lon from cell_centroids.pt.
    """
    hits             = {k: 0 for k in k_values}
    haversine_errors = []
    total            = 0

    for path in shard_paths:
        shard = torch.load(path, map_location='cpu', weights_only=False)
        for ex in shard:
            last_cell = ex['prefix_region_seq'][-1]
            true_dest = ex['dest_region']
            true_lat  = ex['dest_lat']
            true_lon  = ex['dest_lon']

            # Look up predictions; fall back to global if last_cell unseen
            preds = cell_topk.get(last_cell, global_topk)

            for k in k_values:
                if true_dest in preds[:k]:
                    hits[k] += 1

            # Haversine: map top-1 predicted cell → centroid → distance
            pred_lat, pred_lon = centroids[preds[0]]
            haversine_errors.append(
                haversine_km(pred_lat, pred_lon, true_lat, true_lon)
            )
            total += 1

    return {
        'Recall@1'           : hits[1]  / total,
        'Recall@5'           : hits[5]  / total,
        'Recall@10'          : hits[10] / total,
        'Mean Haversine (km)': float(np.mean(haversine_errors)),
        'Med Haversine (km)' : float(np.median(haversine_errors)),
        'n'                  : total,
    }

## Step 4 — Evaluate on validation set

In [5]:
centroids = torch.load(DATA_DIR / 'cell_centroids.pt')

val_paths = sorted((DATA_DIR / 'supervised_shards' / 'val').glob('val_*.pt'))
print(f'Evaluating on {len(val_paths)} val shards...')

val_results = evaluate(val_paths, cell_topk, global_topk, centroids)

print('\nValidation results:')
for k, v in val_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

Evaluating on 11 val shards...

Validation results:
  Recall@1: 0.1204
  Recall@5: 0.2664
  Recall@10: 0.3546
  Mean Haversine (km): 2.2838
  Med Haversine (km): 1.3171
  n: 1062639


## Step 5 — Evaluate on test set (final numbers)

In [6]:
test_paths = sorted((DATA_DIR / 'supervised_shards' / 'test').glob('test_*.pt'))
print(f'Evaluating on {len(test_paths)} test shards...')

test_results = evaluate(test_paths, cell_topk, global_topk, centroids)

print('\nTest results:')
for k, v in test_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

Evaluating on 11 test shards...

Test results:
  Recall@1: 0.1264
  Recall@5: 0.2740
  Recall@10: 0.3623
  Mean Haversine (km): 2.2820
  Med Haversine (km): 1.2991
  n: 1061430


## Results table

In [7]:
r = test_results
print('=' * 72)
print(f'{"Model":<22} {"R@1":>7} {"R@5":>7} {"R@10":>7} {"Mean H (km)":>12} {"Med H (km)":>11}')
print('=' * 72)
print(f'{"Markov Baseline":<22} {r["Recall@1"]:>7.4f} {r["Recall@5"]:>7.4f} {r["Recall@10"]:>7.4f} '
      f'{r["Mean Haversine (km)"]:>12.3f} {r["Med Haversine (km)"]:>11.3f}')
print('=' * 72)
print('(other models to be filled in)')

Model                      R@1     R@5    R@10  Mean H (km)  Med H (km)
Markov Baseline         0.1264  0.2740  0.3623        2.282       1.299
(other models to be filled in)
